# Week 4 — CASE WHEN: Conditional Logic in SQL
## Phase 2b SQL | PORA Academy Cohort 7 — **Demo**

By the end of this session, you will be able to:
- Classify and group data using conditional SQL — write a `CASE WHEN` expression that maps raw column values into higher-level business categories
- Build business categories directly in SQL — bucket a continuous number (payment value) into named bands, then aggregate per band
- Use `SUM(CASE WHEN ... THEN 1 ELSE 0 END)` to count a subset inside a single pass over the table, and turn that count into a correct percentage

---
*Session timing: 2 hours | AI assistance: DeepSeek (introduced today)*

### Run this first

The setup cell below loads all 8 Olist tables into a SQLite database and connects the
`%%sql` magic to it. It is the same cell as Weeks 1–3 — run it once, wait for
`Database ready.`, and leave it alone.

In [ ]:
# =====================================================================
# Olist SQL Setup — runs on BOTH Google Colab and a local machine.
# Run this cell FIRST. It loads the 8 Olist tables into a SQLite
# database and connects the %%sql magic to it. You should not need to
# edit anything unless auto-detection fails (see the two knobs below).
#
# Design notes:
# - We teach SQL with the %%sql cell magic (jupysql), not pd.read_sql().
# - jupysql opens its OWN connection, so the DB must be a real FILE
#   (a :memory: DB would be invisible to it).
# - We use jupysql (the maintained SQL magic). On Colab we install it,
#   because Colab ships the legacy ipython-sql, which (a) can't take a
#   connection by engine variable and (b) renders every result through
#   prettytable.__dict__[style], crashing on modern prettytable with
#   KeyError 'DEFAULT'/'SINGLE_BORDER'. jupysql fixes both.
# - autopandas=True makes every %%sql result a pandas DataFrame, which
#   lets the self-check cells assert on .iloc/.shape directly.
# =====================================================================
import os, glob, sqlite3, tempfile, zipfile
import pandas as pd

# --- Optional knobs (leave blank; only set if auto-detect fails) ------
LOCAL_DATA_DIR = ""   # local run: folder that holds olist_orders_dataset.csv
DRIVE_ZIP_PATH = ""   # Colab: full path to phase-2-python-sql.zip in your Drive
# ---------------------------------------------------------------------

# Detect Colab (google.colab only imports there). Outside Colab — including
# the content-pipeline validator — this falls through to the local branch.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ON_COLAB = True
except ModuleNotFoundError:
    ON_COLAB = False


def _colab_find_zip():
    """Locate phase-2-python-sql.zip in Drive WITHOUT a full recursive scan
    (globbing '/content/drive/MyDrive/**' walks the entire Drive over the
    network and can hang for many minutes). Try explicit paths first, then a
    depth- and count-bounded breadth-first search that prints progress."""
    if DRIVE_ZIP_PATH:
        if os.path.exists(DRIVE_ZIP_PATH):
            return DRIVE_ZIP_PATH
        raise FileNotFoundError(f"DRIVE_ZIP_PATH is set but not found: {DRIVE_ZIP_PATH}")

    target = "phase-2-python-sql.zip"
    # Fast, instant checks of the most likely spots (top of Drive + course folder).
    for cand in (
        f"/content/drive/MyDrive/{target}",
        f"/content/drive/MyDrive/Data Analysis and AI Automation Course Cohort 7/Dataset/{target}",
        f"/content/{target}",
    ):
        if os.path.exists(cand):
            return cand

    # Bounded BFS: depth <= 4, at most ~600 folders, skipping hidden dirs.
    print("Searching your Google Drive for phase-2-python-sql.zip ...")
    root, queue, scanned = "/content/drive/MyDrive", [("/content/drive/MyDrive", 0)], 0
    while queue:
        d, depth = queue.pop(0)
        hit = os.path.join(d, target)
        if os.path.exists(hit):
            return hit
        if depth >= 4:
            continue
        try:
            for e in os.scandir(d):
                if e.is_dir() and not e.name.startswith("."):
                    queue.append((e.path, depth + 1))
        except OSError:
            continue
        scanned += 1
        if scanned % 50 == 0:
            print(f"  ...scanned {scanned} folders")
        if scanned >= 600:
            break

    raise FileNotFoundError(
        "Could not quickly find phase-2-python-sql.zip in your Drive. Put the zip at the "
        "TOP of your Drive (My Drive) and re-run, or set DRIVE_ZIP_PATH at the top of this "
        "cell to its exact path.")


def _find_csv_dir():
    """Return the folder that actually contains olist_orders_dataset.csv."""
    roots = []
    env_dir = os.environ.get("OLIST_DATA_PATH", "")   # set by the pipeline validator
    if env_dir:
        roots.append(env_dir)
    if LOCAL_DATA_DIR:
        roots.append(LOCAL_DATA_DIR)

    if ON_COLAB:
        extract_path = "/content/olist_data"
        # unzip only the first time; reuse the extracted CSVs afterwards
        if not glob.glob(f"{extract_path}/**/olist_orders_dataset.csv", recursive=True):
            zip_path = _colab_find_zip()
            os.makedirs(extract_path, exist_ok=True)
            print(f"Unzipping {os.path.basename(zip_path)} ...")
            with zipfile.ZipFile(zip_path) as z:
                base_dir = os.path.realpath(extract_path)
                for member in z.namelist():
                    member_dest = os.path.realpath(os.path.join(base_dir, member))
                    if os.path.commonpath([base_dir, member_dest]) != base_dir:
                        raise ValueError(f"Unsafe zip member path detected: {member}")
                z.extractall(extract_path)
        roots.append(extract_path)
    else:
        # Local: search cwd (recursively) + a few common spots — never the whole
        # home dir (that recursive walk can be very slow). Set LOCAL_DATA_DIR if
        # your CSVs live elsewhere.
        roots += [os.getcwd(),
                  os.path.expanduser("~/Downloads"),
                  os.path.expanduser("~/Desktop"),
                  os.path.expanduser("~/olist")]

    for root in roots:
        if os.path.exists(os.path.join(root, "olist_orders_dataset.csv")):
            return root
        hits = glob.glob(os.path.join(root, "**", "olist_orders_dataset.csv"), recursive=True)
        if hits:
            return os.path.dirname(hits[0])

    raise FileNotFoundError(
        "Olist CSVs not found. Set LOCAL_DATA_DIR (local) or DRIVE_ZIP_PATH (Colab) at "
        "the top of this cell.")


DATA_DIR = _find_csv_dir()
print("Data folder:", DATA_DIR)

# Build a file-based SQLite DB shared by pandas (loading) and jupysql (querying).
DB_PATH = os.environ.get("OLIST_DB_PATH") or (
    "/content/olist.db" if ON_COLAB else os.path.join(tempfile.gettempdir(), "olist.db"))

tables = {
    "orders": "olist_orders_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "product_category_translation": "product_category_name_translation.csv",
}

conn = sqlite3.connect(DB_PATH)
for table_name, filename in tables.items():
    df = pd.read_csv(os.path.join(DATA_DIR, filename))
    df.to_sql(table_name, conn, if_exists="replace", index=False)
    print(f"Loaded {table_name}: {len(df):,} rows")
conn.close()
print("\nDatabase ready.")

# On Colab, install jupysql so `%load_ext sql` loads it instead of the legacy
# ipython-sql (see header). Off Colab (local / pipeline validator) jupysql is
# already installed, so we skip the install and stay offline-safe.
if ON_COLAB:
    get_ipython().run_line_magic("pip", "install --quiet --upgrade jupysql")

get_ipython().run_line_magic("load_ext", "sql")

# Guard: if the legacy ipython-sql was already loaded earlier THIS session (e.g.
# an older cell ran first), the freshly installed jupysql cannot hot-swap in — a
# runtime restart is the only fix. jupysql exposes sql.connection.ConnectionManager;
# ipython-sql does not. Stop with a clear instruction instead of a later cryptic
# prettytable KeyError.
import sql.connection as _sqlconn
if not hasattr(_sqlconn, "ConnectionManager"):
    raise RuntimeError(
        "Legacy ipython-sql is active, not jupysql. On Colab: Runtime -> Restart session, "
        "then run THIS setup cell first (before any other cell). Locally: "
        "pip install --upgrade jupysql and restart the kernel."
    )

# Connect the %%sql magic to the SAME database file. autopandas=True is REQUIRED
# (see header). We connect with run_line_magic (not a literal `%sql` line) so the
# computed DB_PATH is interpolated correctly. Do NOT set SqlMagic.style.
get_ipython().run_line_magic("config", "SqlMagic.autopandas = True")
get_ipython().run_line_magic("config", "SqlMagic.feedback = 0")
get_ipython().run_line_magic("sql", f"sqlite:///{DB_PATH}")

# Verify (expected row counts — do not alter without re-running against data):
#   orders 99,441 | customers 99,441 | order_items 112,650 | order_payments 103,886
#   order_reviews 99,224 | products 32,951 | sellers 3,095 | product_category_translation 71


## Why this matters

The `orders` table records an `order_status` for each of its 99,441 orders, and it uses
eight different values to do it: `delivered`, `shipped`, `canceled`, `unavailable`,
`invoiced`, `processing`, `created`, `approved`. That level of detail is exactly what
the operations team needs. It is exactly what a board meeting does not.

When Olist's management asks *"what proportion of our orders actually completed?"* they
are not asking for eight numbers. They are asking for three or four: how many finished,
how many are still moving through the pipeline, how many died. Nowhere in the database
is there a column holding that answer — `status_group` does not exist as data. It
exists only as a **rule** the business applies to the data.

`CASE WHEN` is how you write that rule in SQL. It builds a new column on the fly, row by
row, out of a condition you define — so you can classify 99,441 orders into business
categories without editing a single stored value, and then `GROUP BY` the category you
just invented.

## 1. `CASE WHEN` — turning raw values into business categories

A `CASE` expression is SQL's version of "if this, then that". You give it a list of
`WHEN <condition> THEN <value>` branches and it walks them **in order, top to bottom**,
for every single row. The moment a condition is true, `CASE` returns that branch's value
and stops looking — the remaining branches are never even evaluated for that row. If no
branch matches, the `ELSE` value is returned instead.

Think of a postal sorting office. Each parcel comes past one clerk who checks a short
list of rules in a fixed order — "is it going to Lagos? then bin one; is it going
anywhere else in the South West? then bin two; otherwise, bin three" — and drops the
parcel in the first bin whose rule fits. `CASE WHEN` is that clerk, and the bins are
the categories you named.

The whole expression sits **where a column would normally sit**, in the `SELECT` list,
and you give it a name with `AS` just like any other column. That name is the important
part: once the category has a name, every clause you already know — `WHERE`,
`GROUP BY`, `ORDER BY` — can work with it as if it had been in the table all along.

In [ ]:
%%sql
-- Classify raw order statuses into business categories. Keeping order_status
-- alongside status_group lets you SEE the mapping the CASE expression applied.
-- Expected: delivered/Completed 96,478 | shipped/In Progress 1,107
--           canceled/Canceled 625 | unavailable/Other 609
--           invoiced/In Progress 314 | processing/In Progress 301
SELECT order_status,
       CASE
           WHEN order_status = 'delivered' THEN 'Completed'
           WHEN order_status IN ('shipped', 'invoiced', 'processing', 'approved') THEN 'In Progress'
           WHEN order_status = 'canceled' THEN 'Canceled'
           ELSE 'Other'
       END AS status_group,
       COUNT(*) AS count
FROM orders
GROUP BY order_status
ORDER BY count DESC

## 2. Grouping *by* the category you just invented

The query above still returns one row per raw status — useful for checking your work,
useless for a board slide. To get the summary the business actually asked for, group by
`status_group` instead of `order_status`. The eight raw statuses collapse into four
business buckets.

Notice what SQLite lets you do here: `GROUP BY status_group` refers to the **alias** of
the `CASE` expression, not the expression itself. That is a convenience SQLite (and
several other databases) offers; if you ever meet a database that rejects it, the fix is
to paste the entire `CASE ... END` block into the `GROUP BY` as well. Both forms mean the
same thing.

One row of this result is now a *category*, not a status — so `COUNT(*)` counts the
orders that fell into that category, which is exactly the number management wanted.

In [ ]:
%%sql
-- The board-slide version: four business buckets instead of eight raw statuses.
-- 'delivered' is the only status mapping to Completed, so that row must read
-- 96,478 — the delivered-order count you have used since Week 1.
SELECT CASE
           WHEN order_status = 'delivered' THEN 'Completed'
           WHEN order_status IN ('shipped', 'invoiced', 'processing', 'approved') THEN 'In Progress'
           WHEN order_status = 'canceled' THEN 'Canceled'
           ELSE 'Other'
       END AS status_group,
       COUNT(*) AS order_count   -- Expected on the Completed row: 96,478
FROM orders
GROUP BY status_group
ORDER BY order_count DESC

## 3. Bucketing a continuous number — payment value bands

The status example mapped one set of labels onto another. The far more common analytical
use of `CASE WHEN` is **binning a continuous number**: turning a column of thousands of
distinct values into a handful of named ranges you can actually reason about.

`order_payments.payment_value` is a REAL in Brazilian reais, and it takes on tens of
thousands of different values. "What is our average payment?" is answerable but shallow.
"How much of our volume is small-basket versus premium, and what does the average look
like *within* each band?" is a question that changes what the business does — and it
needs bands that do not exist in the data.

Here the **order of the `WHEN` branches is doing real work**. Because the first true
branch wins, you only ever need to state the upper bound of each band: by the time SQL
evaluates `WHEN payment_value < 200`, every row below 50 has already been claimed by the
branch above it, so that branch effectively means "between 50 and 200". Write the bands
from smallest to largest and the logic stays this clean.

One grain warning before you read the result: `order_payments` holds **103,886 rows for
99,441 orders** — an order split across a voucher and a credit card contributes two rows.
So `COUNT(*)` below counts *payment records*, not orders. That is the honest reading of
this table, and naming things precisely is half of analytical work.

In [ ]:
%%sql
-- Bin a continuous column into named bands. Bands are checked in order, so each
-- WHEN only needs its upper bound. ORDER BY avg_value puts the bands in
-- ascending order of value. (Each row = payment records, not orders.)
SELECT
    CASE
        WHEN payment_value < 50 THEN 'Low (< R$50)'
        WHEN payment_value < 200 THEN 'Mid (R$50–200)'
        WHEN payment_value < 500 THEN 'High (R$200–500)'
        ELSE 'Premium (R$500+)'
    END AS value_category,
    COUNT(*) AS count,
    ROUND(AVG(payment_value), 2) AS avg_value
FROM order_payments
GROUP BY value_category
ORDER BY avg_value

---
## 🤖 New this week: AI assistance with DeepSeek

From today you may use DeepSeek to help draft SQL. Three weeks of writing queries by
hand were the prerequisite — not because AI is off-limits, but because an AI-drafted
query is only useful to someone who can tell whether it is **right**. SQL fails quietly:
a query with a subtly wrong `CASE` order or a missing `ELSE` does not error, it just
returns a confident, wrong table.

**The prompt-then-verify protocol — every time, no exceptions:**

1. **Say what you want in plain English first.** If you cannot state the question, you
   are not ready to prompt. Write it down, then prompt.
2. **Give DeepSeek the context it cannot guess.** Name the table, the exact columns, the
   SQL dialect (SQLite), and the shape of answer you want. "Write SQL for orders" gets
   you invented column names.
3. **Run the query yourself and check the number.** Compare it against a value this
   curriculum has verified. If the number does not match, the query is wrong — not the
   curriculum.
4. **Be able to explain every line.** If a branch of a generated `CASE` is unclear, ask
   DeepSeek to explain it *before* you use the result. You will be asked to explain your
   queries in class.

**A good prompt for today's material:**

```
I have a SQLite table `orders` with a text column `order_status`
(values: delivered, shipped, canceled, unavailable, invoiced, processing,
created, approved).
Write a query that counts how many orders are 'delivered', using
SUM(CASE WHEN ...) so the result is a single row with one column.
```

Run that prompt now. Whatever DeepSeek returns, paste it in and run it — then check the
number against the cell below, which is the verification step of the protocol. Anything
other than **96,478** means the generated query is wrong.

In [ ]:
%%sql
-- Step 3 of the protocol: run it, and check the number before you trust it.
-- SUM(CASE WHEN <cond> THEN 1 ELSE 0 END) scores each row 1 or 0 and adds the
-- scores up — a count of a subset, computed in one pass over the table.
SELECT SUM(CASE WHEN order_status = 'delivered' THEN 1 ELSE 0 END) AS completed_orders  -- Expected: 96,478
FROM orders

## Going deeper — conditional counting, and the integer-division trap

That `SUM(CASE WHEN ... THEN 1 ELSE 0 END)` pattern is worth dwelling on, because it is
how you put *several different filters side by side in one row*. A `WHERE` clause filters
the whole query — one filter, one number. A conditional `SUM` filters per column, so you
can return the delivered count, the canceled count and the total from a single scan.

That naturally leads to percentages, and percentages are where SQLite lays a trap.
`625 * 100 / 99441` looks like arithmetic that should give 0.63. It gives **0**. Both
operands are integers, so SQLite performs integer division and throws away everything
after the decimal point. Nothing errors; the report just says a flat zero.

The fix is to make one operand a REAL, and the conventional way to do that is to
multiply by `1.0`. Run the cell and look at `pct_wrong` beside `pct_right` — same
inputs, and the untreated version has silently deleted the entire answer.

In [ ]:
%%sql
-- Conditional counting + the integer-division trap, side by side.
SELECT
    SUM(CASE WHEN order_status = 'canceled' THEN 1 ELSE 0 END) AS canceled_orders,  -- Expected: 625
    COUNT(*)                                                   AS total_orders,     -- Expected: 99,441
    -- WRONG: integer / integer truncates toward zero — this returns 0
    SUM(CASE WHEN order_status = 'canceled' THEN 1 ELSE 0 END) * 100 / COUNT(*) AS pct_wrong,
    -- RIGHT: * 1.0 forces REAL division before the percentage is taken
    ROUND(SUM(CASE WHEN order_status = 'canceled' THEN 1 ELSE 0 END) * 1.0 * 100 / COUNT(*), 2) AS pct_right
FROM orders

## Common mistakes

**Mistake 1 — omitting `ELSE`.** A `CASE` with no `ELSE` returns `NULL` for any row that
matched no branch. The rows do not vanish; they collect in a `NULL` bucket in your
`GROUP BY` output that you never asked for and may not notice. Always write the `ELSE`,
even if it is only `ELSE 'Other'` — it forces you to decide what the leftovers are.

**Mistake 2 — comparing to `NULL` with `=`.** Inside a `CASE`, `WHEN some_column = NULL`
is not false, it is *unknown* — which is never true, so the branch can never fire. The
missing-date rows then fall through to your `ELSE` and get mislabelled. The only way to
test for a missing value is `IS NULL` / `IS NOT NULL`.

**Mistake 3 — writing the bands in the wrong order.** Because the first true branch wins,
putting a wide band before a narrow one swallows the narrow one whole. `WHEN payment_value
< 500 THEN 'High'` placed above `WHEN payment_value < 200 THEN 'Mid'` means no payment is
ever labelled `'Mid'` — every one under 200 is also under 500, so it is claimed first. The
`'Mid'` band silently reports zero rows, and nothing warns you.

The correct query below fixes mistakes 1 and 2 together: an explicit `ELSE`, and `IS NULL`
to catch orders that have no delivery date recorded.

In [ ]:
%%sql
-- ── COMMON MISTAKES ────────────────────────────────────────────────
-- WRONG 1 — no ELSE: every unmatched row becomes NULL and forms a bucket
-- you never asked for:
--   CASE WHEN order_status = 'delivered' THEN 'Completed' END AS status_group
--
-- WRONG 2 — '= NULL' is never true, so this branch can NEVER fire:
--   WHEN order_delivered_customer_date = NULL THEN 'Never delivered'
--
-- WRONG 3 — wide band first swallows the narrow one; 'Mid' returns 0 rows:
--   CASE WHEN payment_value < 500 THEN 'High'
--        WHEN payment_value < 200 THEN 'Mid' ... END
--
-- CORRECT — explicit ELSE, and IS NULL for the missing-date branch.
-- Only the Completed row carries a curriculum-verified count (96,478); the
-- other buckets are shown to make the classification visible, not asserted.
SELECT CASE
           WHEN order_status = 'delivered' THEN 'Completed'
           WHEN order_delivered_customer_date IS NULL THEN 'Never delivered'
           ELSE 'Other'
       END AS delivery_state,
       COUNT(*) AS order_count   -- Expected on the Completed row: 96,478
FROM orders
GROUP BY delivery_state
ORDER BY order_count DESC

## Mini-challenge — your turn

⏱ ~5–10 minutes

Management wants the status summary **with a share column**: for each of the four
business buckets (`Completed` / `In Progress` / `Canceled` / `Other`), how many orders,
and what percentage of all orders is that?

Build it from the pieces you have used above:

1. The `CASE WHEN` from section 2 to produce `status_group`.
2. `COUNT(*)` for the bucket size.
3. A percentage over the whole table. `COUNT(*)` inside a `GROUP BY` gives you the
   *bucket's* count — for the denominator you need the total, which is
   `(SELECT COUNT(*) FROM orders)`. Remember `* 1.0`, and wrap the whole thing in
   `ROUND(..., 2)`.

**Expected:** four rows. The `Completed` row must read **96,478** orders — the delivered
count you have checked since Week 1 — and its share must be a little over 97%, not `97`
flat and certainly not `0`. If any share comes back as a whole number, you dropped the
`* 1.0`.

*Stretch, if you finish early:* add `ROUND(AVG(...), 2)` of something meaningful per
bucket, or re-run the payment-band query from section 3 with the bands deliberately in
the wrong order and confirm for yourself that the `'Mid'` band disappears.

In [ ]:
%%sql
-- ⏱ ~5-10 min — your turn! Replace the placeholder below with your own query.
SELECT 'write your query here' AS todo

## Session Summary

| Clause / idea | What it does | Example |
|---|---|---|
| `CASE WHEN ... THEN ... END` | builds a new column from a rule, row by row | `CASE WHEN order_status = 'delivered' THEN 'Completed' ... END` |
| `ELSE` | the value for rows that matched no `WHEN` — without it they become `NULL` | `ELSE 'Other'` |
| `AS` on a `CASE` | names the invented column so other clauses can use it | `END AS status_group` |
| `GROUP BY <case alias>` | aggregates by the category you just invented | `GROUP BY status_group` |
| `WHEN ... IN (...)` | one branch covering several values | `WHEN order_status IN ('shipped', 'invoiced') THEN 'In Progress'` |
| band ordering | first true branch wins — write narrow bands first | `WHEN v < 50 ... WHEN v < 200 ...` |
| `SUM(CASE WHEN c THEN 1 ELSE 0 END)` | counts a subset without a `WHERE` — several filters in one row | `SUM(CASE WHEN order_status = 'canceled' THEN 1 ELSE 0 END)` |
| `* 1.0` | forces REAL division so a percentage is not truncated to an integer | `cnt * 1.0 * 100 / COUNT(*)` |
| `IS NULL` inside `CASE` | the only way to test a missing value (`= NULL` never fires) | `WHEN order_delivered_customer_date IS NULL THEN ...` |

**The two questions to ask about every `CASE` you write:** what happens to a row that
matches nothing — and is my branch order sending rows to the first bin that fits, or the
one I meant?

---
**Coming up Thursday**: **date functions**. Olist stores every timestamp as *text*, so
`order_purchase_timestamp` is a string until you convert it. You will use
`strftime()` to pull the year and month out of those strings — revealing the November
2017 Black Friday spike of 7,544 orders — and `julianday()` to subtract two dates and
get Olist's average delivery time of **12.6 days**, then count the **7,826** orders that
arrived later than their estimated delivery date. Today's `CASE WHEN` comes with you:
combined with date arithmetic it is how you classify a delivery as fast, standard or
slow.

Thursday closes with a **group exercise** built on both days' material: classifying
customers into geographic regions with `CASE WHEN`, comparing weekend vs. weekday order
volume with `strftime('%w', ...)`, ranking the fastest and slowest delivery states, and
building a month-by-month 2018 summary with a "peak month" flag.